# J1S1 — Setup & GitHub Copilot
**Formation Data Science · Jour 1 · Session 1 · 09h00 – 10h30**

> **Organisation GitHub :** `bankrisk-formation`  
> **Repo de référence formateur :** `alexdiby/bankrisk-credit`  
> **Workflow :** Fork de l'org → Clone du fork → Développe → Push sur origin → PR vers l'org  
> **Dataset :** `credit_risk_dataset.csv` (32 581 lignes · 12 colonnes · CC0)

---

## 🎯 Objectifs

- Forker le repo de l'organisation `bankrisk-formation` vers son compte GitHub
- Cloner le fork en local et configurer les remotes `origin` et `upstream`
- Mettre en place la structure de projet standard avec `requirements.txt` et `.gitignore`
- Configurer GitHub Copilot et pratiquer les 3 modes (complétion, inline, chat latéral)
- Charger le dataset fil rouge et vérifier le taux de défaut : **21,8 %**
- Générer le `README.md` et merger une PR vers l'organisation


---

### Dataset : Credit Risk Kaggle (CC0)

| Colonne | Type | Description |
|---------|------|-------------|
| `person_age` | int | Âge du demandeur |
| `person_income` | float | Revenu annuel ($) |
| `loan_amnt` | float | Montant du crédit |
| `loan_int_rate` | float | Taux d'intérêt (%) |
| **`loan_status`** | **int** | **CIBLE : 0=sain · 1=défaut** |
| `loan_grade` | str | Grade de risque A–G |
| `loan_percent_income` | float | Ratio endettement |
| `cb_person_default_on_file` | str | Historique défaut Y/N |

> **Source** : https://www.kaggle.com/datasets/laotse/credit-risk-dataset  
> **Licence** : CC0 (domaine public — utilisation libre)


In [1]:
from pathlib import Path
import os

# ── Racine du projet ─────────────────────────────────────────────────────────
# Le notebook est dans notebooks/ — on remonte d'un niveau pour être à la racine
# Cette cellule doit être exécutée EN PREMIER à chaque session
ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent

# Fallback Colab : si le notebook est lancé depuis la racine du repo
if not (ROOT / "requirements.txt").exists() and (Path.cwd() / "requirements.txt").exists():
    ROOT = Path.cwd()

os.chdir(ROOT)

print(f"Racine du projet : {ROOT}")
print(f"Répertoire de travail : {Path.cwd()}")
print()

# Vérification de l'arborescence attendue
expected = ["data/raw", "data/processed", "notebooks", "src", "tests"]
for folder in expected:
    status = "✅" if (ROOT / folder).exists() else "⚠️  à créer"
    print(f"  {status}  {folder}/")


Racine du projet : d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation
Répertoire de travail : d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation

  ✅  data/raw/
  ✅  data/processed/
  ✅  notebooks/
  ✅  src/
  ✅  tests/


---

## Bloc 1 — Vérification de l'environnement

> **💻 Deux options d'environnement — au choix du participant :**
>
> | | VS Code (local) | Google Colab |
> |-|-----------------|---------------|
> | **Avantage** | Contrôle total, Git intégré, Copilot natif | Zéro installation, GPU gratuit |
> | **Inconvénient** | Installation Python + extensions requises | Session non persistante |
> | **Recommandé pour** | Usage professionnel long terme | Démarrage rapide / contrainte matérielle |
>
> Les blocs suivants fonctionnent dans les **deux** environnements.


In [2]:
import sys
print(f"Python  : {sys.version.split()[0]}")
print(f"Chemin  : {sys.executable}")

import pandas as pd
import numpy as np

print(f"Pandas  : {pd.__version__}")
print(f"NumPy   : {np.__version__}")
print()

# Détecter l'environnement
if '.venv' in sys.executable:
    print("✅ Virtualenv .venv actif (VS Code)")
elif 'google.colab' in sys.modules:
    print("✅ Environnement Google Colab détecté")
else:
    print("⚠️  Attention : environnement non reconnu")
    print(f"   Interpréteur actif : {sys.executable}")


Python  : 3.10.11
Chemin  : d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation\.venv\Scripts\python.exe
Pandas  : 2.3.3
NumPy   : 2.2.6

✅ Virtualenv .venv actif (VS Code)


### 💡 Résolution des erreurs courantes

**`ModuleNotFoundError` sous VS Code**
1. Cliquer **Select Kernel** en haut à droite du notebook
2. Choisir **.venv (Python 3.10.x)**
3. Si absent → **Ctrl+Shift+P** → `Python: Select Interpreter` → `.venv`

**Google Colab : installer les dépendances de la session**
```python
!pip install pandas numpy plotly scikit-learn -q
```


---

## Bloc 2 — Workflow Organisation → Fork → Clone

### Le modèle mental

```
GitHub                                              Local (VS Code / Colab)
─────────────────────────────────────────────────   ───────────────────────
bankrisk-formation/credit-risk-formation  ──fork──▶  TON_USERNAME/credit-risk-formation
        ▲  (upstream)                                        │  (origin)
        │                                                    │  git clone
        │  Pull Request                                      ▼
        └────────────────────────────────────────── ton ordinateur
```

**Règle :** on travaille sur `origin` (son fork), on contribue à `upstream` (l'org) via une PR.

---

### Étapes dans le terminal VS Code

```bash
# 1. Configuration Git (une seule fois par machine)
git config --global user.name "Ton Prénom Nom"
git config --global user.email "ton@email.com"

# 2. Sur github.com/bankrisk-formation/credit-risk-formation
#    → Fork (bouton en haut à droite) → Create fork
#    Résultat : TON_USERNAME/credit-risk-formation

# 3. Cloner le FORK (pas le repo de l'org !)
git clone https://github.com/TON_USERNAME/credit-risk-formation.git
cd credit-risk-formation

# 4. Connecter l'organisation comme upstream
git remote add upstream https://github.com/bankrisk-formation/credit-risk-formation.git

# 5. Vérifier les deux remotes
git remote -v
# origin    https://github.com/TON_USERNAME/credit-risk-formation.git
# upstream  https://github.com/bankrisk-formation/credit-risk-formation.git
```

> **Google Colab :** Git n'est pas disponible nativement.  
> Créer le fork sur github.com, puis cloner depuis un terminal local ou utiliser le terminal intégré de Colab (`!git clone ...`).


In [68]:
# ── Vérification des remotes depuis le notebook ─────────────────────────────
import subprocess

def git_run(cmd):
    """Exécute une commande git et affiche le résultat."""
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
    return result.returncode

print("=== Remotes configurés ===")
ret = git_run("git remote -v")
print()
if ret != 0:
    print("⚠️  Git non initialisé — exécuter les étapes bash ci-dessus d'abord")
else:
    out = subprocess.run("git remote -v", shell=True, capture_output=True, text=True).stdout
    if "origin" in out and "upstream" in out:
        print("✅ origin (fork) et upstream (org) correctement configurés")
    elif "origin" in out:
        print("⚠️  origin OK — upstream manquant")
        print("   Exécuter : git remote add upstream https://github.com/bankrisk-formation/credit-risk-formation.git")
    else:
        print("❌ Aucun remote — reprendre les étapes du Bloc 2")


=== Remotes configurés ===
$ git remote -v
origin	https://github.com/alexdiby/credit-risk-formation.git (fetch)
origin	https://github.com/alexdiby/credit-risk-formation.git (push)
upstream	https://github.com/bankrisk-formation/credit-risk-formation.git (fetch)
upstream	https://github.com/bankrisk-formation/credit-risk-formation.git (push)


✅ origin (fork) et upstream (org) correctement configurés


---

## Bloc 3 — Structure du projet

Avant de créer le repo GitHub, on définit la structure locale standard du projet.  
Cette arborescence sera identique pour **toutes les sessions de la formation**.

```
credit-risk-<TON_USERNAME>/
├── data/
│   ├── raw/            ← Dataset original — jamais modifié
│   └── processed/      ← Données nettoyées et transformées
├── notebooks/          ← Notebooks Jupyter par session
├── src/                ← Scripts Python réutilisables
├── tests/              ← Tests unitaires (pytest)
├── requirements.txt    ← Dépendances du projet
└── .gitignore          ← Fichiers exclus du contrôle de version
```

> **Règle absolue :** `data/raw/` n'est **jamais modifié** après dépôt initial.  
> Toutes les transformations produisent des fichiers dans `data/processed/`.


In [3]:
# ── Définition de la structure ──────────────────────────────────────────────
# ROOT est défini dans la cellule d'initialisation ci-dessus
folders = [
    "data/raw",        # Dataset original — jamais modifié
    "data/processed",  # Données nettoyées
    "notebooks",       # Notebooks Jupyter
    "src",             # Scripts Python
    "tests",           # Tests unitaires
]

# ── Création des dossiers + .gitkeep à la RACINE du projet ──────────────────
for folder in folders:
    path = ROOT / folder
    path.mkdir(parents=True, exist_ok=True)
    gitkeep = path / ".gitkeep"
    if not gitkeep.exists():
        gitkeep.touch()
    print(f"✅ {ROOT}/{folder}/")

print()
print("Structure créée à la racine du projet.")
print(f"⚠️  Copie le fichier credit_risk_dataset.csv dans {ROOT}/data/raw/")


✅ d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation/data/raw/
✅ d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation/data/processed/
✅ d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation/notebooks/
✅ d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation/src/
✅ d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation/tests/

Structure créée à la racine du projet.
⚠️  Copie le fichier credit_risk_dataset.csv dans d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation/data/raw/


### Pourquoi les `.gitkeep` ?

Git ne traque pas les dossiers vides.  
Un fichier `.gitkeep` (vide, par convention) permet de :
- Committer la structure du projet dès le départ
- Garantir que `data/raw/` et `data/processed/` existent dans le repo **sans** y inclure les données

Le `.gitignore` exclut le contenu de ces dossiers, mais préserve les `.gitkeep` via une règle d'exception `!data/raw/.gitkeep`.


---

## Bloc 4 — Génération du `requirements.txt`

Le fichier `requirements.txt` liste toutes les dépendances Python du projet avec leurs versions minimales.  
Il permet à n'importe quel collaborateur de recréer l'environnement exact avec :

```bash
pip install -r requirements.txt
```

Les dépendances couvrent les **3 jours de formation** — certaines ne seront utilisées qu'à partir de J2 ou J3.


In [4]:
requirements_content = (
    "# ── Core data science ──────────────────────────────────────────\n"
    "pandas>=2.0.0\n"
    "numpy>=1.24.0\n\n"
    "# ── Visualisation ──────────────────────────────────────────────\n"
    "plotly>=5.18.0\n\n"
    "# ── Machine learning ───────────────────────────────────────────\n"
    "scikit-learn>=1.3.0\n\n"
    "# ── MLOps ──────────────────────────────────────────────────────\n"
    "mlflow>=2.10.0\n"
    "dagshub>=0.3.0\n\n"
    "# ── LLM / Hugging Face ─────────────────────────────────────────\n"
    "huggingface-hub>=0.20.0\n"
    "requests>=2.31.0\n\n"
    "# ── Data engineering ───────────────────────────────────────────\n"
    "pyspark>=3.5.0\n"
    "delta-spark>=3.1.0\n\n"
    "# ── App ────────────────────────────────────────────────────────\n"
    "streamlit>=1.32.0\n\n"
    "# ── Notebook / utils ───────────────────────────────────────────\n"
    "ipykernel>=6.29.0\n"
    "jupyter>=1.0.0\n"
    "python-dotenv>=1.0.0\n"
)

# Écriture à la RACINE du projet (pas dans notebooks/)
with open(ROOT / "requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_content)

print("✅ requirements.txt généré")
print()
print(requirements_content)


✅ requirements.txt généré

# ── Core data science ──────────────────────────────────────────
pandas>=2.0.0
numpy>=1.24.0

# ── Visualisation ──────────────────────────────────────────────
plotly>=5.18.0

# ── Machine learning ───────────────────────────────────────────
scikit-learn>=1.3.0

# ── MLOps ──────────────────────────────────────────────────────
mlflow>=2.10.0
dagshub>=0.3.0

# ── LLM / Hugging Face ─────────────────────────────────────────
huggingface-hub>=0.20.0
requests>=2.31.0

# ── Data engineering ───────────────────────────────────────────
pyspark>=3.5.0
delta-spark>=3.1.0

# ── App ────────────────────────────────────────────────────────
streamlit>=1.32.0

# ── Notebook / utils ───────────────────────────────────────────
ipykernel>=6.29.0
jupyter>=1.0.0
python-dotenv>=1.0.0



---

### Installation des dépendances

Maintenant que le `requirements.txt` est créé, on installe toutes les dépendances dans le venv actif.

> ⚠️ **Cette cellule doit être exécutée une seule fois par environnement.**  
> Sous VS Code : le kernel `.venv` doit être actif.  
> Sous Colab : l'installation est temporaire — à relancer si la session redémarre.


In [5]:
import subprocess, sys

# ── Installation depuis requirements.txt ────────────────────────────────────
print("Installation des dépendances...")
print(f"Interpréteur : {sys.executable}")
print()

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r",
     str(ROOT / "requirements.txt"), "-q", "--no-warn-script-location"],
    capture_output=True, text=True
)

if result.returncode == 0:
    print("✅ Toutes les dépendances installées avec succès")
else:
    print("⚠️  Erreur lors de l'installation :")
    print(result.stderr[-1000:])  # dernières 1000 chars de l'erreur

# ── Vérification des packages critiques ─────────────────────────────────────
packages = {
    "pandas":    "Manipulation de données",
    "numpy":     "Calcul vectorisé",
    "plotly":    "Visualisation (J1S3)",
    "sklearn":   "Machine learning (J2S2)",
    "pyarrow":   "Format Parquet (J1S2)",
    "mlflow":    "Experiment tracking (J2S3)",
}

print()
print("Vérification des packages :")
for pkg, usage in packages.items():
    try:
        mod = __import__(pkg if pkg != "sklearn" else "sklearn")
        version = getattr(mod, "__version__", "?")
        print(f"  ✅ {pkg:<12} {version:<10}  — {usage}")
    except ImportError:
        print(f"  ❌ {pkg:<12} {'manquant':<10}  — {usage}")

print()
print("💡 Les packages J3 (pyspark, delta-spark, streamlit) seront installés en J3S1.")


Installation des dépendances...
Interpréteur : d:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation\.venv\Scripts\python.exe

✅ Toutes les dépendances installées avec succès

Vérification des packages :
  ✅ pandas       2.3.3       — Manipulation de données
  ✅ numpy        2.2.6       — Calcul vectorisé
  ✅ plotly       6.8.0       — Visualisation (J1S3)
  ✅ sklearn      1.7.2       — Machine learning (J2S2)
  ✅ pyarrow      24.0.0      — Format Parquet (J1S2)
  ✅ mlflow       3.14.0      — Experiment tracking (J2S3)

💡 Les packages J3 (pyspark, delta-spark, streamlit) seront installés en J3S1.


### Correspondance libs / sessions

| Librairie | Utilisée à partir de |
|-----------|---------------------|
| `pandas`, `numpy` | J1S1 — dès aujourd'hui |
| `plotly` | J1S3 — EDA & Visualisation |
| `scikit-learn` | J2S2 — Régression logistique |
| `mlflow`, `dagshub` | J2S3 — Experiment tracking |
| `huggingface-hub` | J2S4 — LLM Explanations |
| `pyspark`, `delta-spark` | J3S2 — PySpark ETL |
| `streamlit` | J3S1 — Dashboard Streamlit |
| `python-dotenv` | J2S3 — gestion des tokens API |


---

## Bloc 5 — Génération du `.gitignore`

Le `.gitignore` définit les fichiers que Git doit **ignorer** lors des commits et pull requests.

**Principe fondamental du projet :** les données ne sont jamais commitées dans le repo.
- `data/raw/*` → données sources brutes (potentiellement volumineuses)
- `data/processed/*` → fichiers intermédiaires générés (`.parquet`, `.csv`)

Les `.gitkeep` sont **explicitement conservés** via des règles d'exception (`!`) pour maintenir la structure de dossiers.


In [6]:
gitignore_content = (
    "# ── Données — ignorées dans les PR ────────────────────────────\n"
    "data/raw/*\n"
    "data/processed/*\n"
    "!data/raw/.gitkeep\n"
    "!data/processed/.gitkeep\n"
    "*.csv\n"
    "*.parquet\n"
    "*.xlsx\n\n"
    "# ── Outputs modèles ────────────────────────────────────────────\n"
    "*.pkl\n"
    "*.joblib\n"
    "mlruns/\n\n"
    "# ── Python ─────────────────────────────────────────────────────\n"
    "__pycache__/\n"
    "*.py[cod]\n"
    "*.egg-info/\n"
    "dist/\n"
    "build/\n"
    ".venv/\n"
    "env/\n"
    "venv/\n\n"
    "# ── Jupyter ────────────────────────────────────────────────────\n"
    ".ipynb_checkpoints/\n\n"
    "# ── Secrets ────────────────────────────────────────────────────\n"
    ".env\n"
    ".env.*\n"
    "secrets.yaml\n\n"
    "# ── OS ─────────────────────────────────────────────────────────\n"
    ".DS_Store\n"
    "Thumbs.db\n"
)

# Écriture à la RACINE du projet (pas dans notebooks/)
with open(ROOT / ".gitignore", "w", encoding="utf-8") as f:
    f.write(gitignore_content)

print("✅ .gitignore généré")
print()
print(gitignore_content)


✅ .gitignore généré

# ── Données — ignorées dans les PR ────────────────────────────
data/raw/*
data/processed/*
!data/raw/.gitkeep
!data/processed/.gitkeep
*.csv
*.parquet
*.xlsx

# ── Outputs modèles ────────────────────────────────────────────
*.pkl
*.joblib
mlruns/

# ── Python ─────────────────────────────────────────────────────
__pycache__/
*.py[cod]
*.egg-info/
dist/
build/
.venv/
env/
venv/

# ── Jupyter ────────────────────────────────────────────────────
.ipynb_checkpoints/

# ── Secrets ────────────────────────────────────────────────────
.env
.env.*
secrets.yaml

# ── OS ─────────────────────────────────────────────────────────
.DS_Store
Thumbs.db



### Explication des règles clés

| Règle | Effet |
|-------|-------|
| `data/raw/*` | Ignore tout le contenu de `data/raw/` |
| `!data/raw/.gitkeep` | **Exception** : conserve le `.gitkeep` pour préserver la structure |
| `*.parquet` | Ignore tous les fichiers Parquet générés au fil des sessions |
| `mlruns/` | Ignore les logs MLflow locaux — l'expérimentation est tracée sur DagsHub |
| `.env` | Protège les tokens API (Hugging Face, DagsHub) — **ne jamais committer** |
| `.ipynb_checkpoints/` | Ignore les sauvegardes automatiques de Jupyter |


---

## Bloc 6 — GitHub Copilot : exercices progressifs : exercices progressifs

### Activation (VS Code)

1. [github.com/features/copilot](https://github.com/features/copilot) → **Start using Copilot for free**
2. VS Code : extension **GitHub Copilot** → Sign in to GitHub
3. L'icône Copilot en bas de VS Code doit être active (non barrée)

> **Google Colab :** Copilot n'est pas disponible nativement.  
> Utiliser **Claude.ai** comme assistant IA pendant la formation.

---

### Raccourcis essentiels (VS Code)

| Raccourci | Action |
|-----------|--------|
| `Tab` | Accepter la suggestion |
| `Échap` | Refuser |
| `Alt + ]` | Suggestion suivante |
| `Ctrl + I` | Chat inline — modifier/expliquer le code sélectionné |
| `Ctrl + Alt + I` | Chat latéral — questions complexes, refactoring |

> **Règle d'or :** comprendre avant d'accepter — Copilot peut générer du code incorrect.

---

### Les 3 modes à connaître

| Mode | Déclencheur | Usage typique |
|------|-------------|---------------|
| **Complétion** | Taper un commentaire + attendre 2 sec | Générer du code courant |
| **Chat inline** | Sélectionner du code + `Ctrl+I` | Expliquer, modifier, corriger |
| **Chat latéral** | `Ctrl+Alt+I` | Architecture, questions complexes |


---

### 🧪 Exercice 1 — Technique du commentaire-intention

**Principe :** écrire ce qu'on veut faire en commentaire → attendre 2 secondes → `Tab`.

**Consigne :** taper **uniquement** les commentaires dans la cellule ci-dessous, un par un.  
Laisser Copilot compléter chaque ligne de code avant de passer au commentaire suivant.

> ⚠️ Ne pas copier-coller le code — l'objectif est de voir Copilot le générer en temps réel.


In [ ]:
import pandas as pd
import numpy as np

# Charger le dataset credit risk depuis le fichier CSV
df = pd.read_csv(ROOT / "data/raw/credit_risk_dataset.csv")

# Afficher le nombre de lignes et colonnes
print(f"Shape : {df.shape}")

# Calculer le taux de défaut moyen
taux = df["loan_status"].mean()
print(f"Taux de défaut : {taux:.1%}")

# Afficher les 3 premières lignes
df.head(3)


---

### 🧪 Exercice 2 — Chat inline `Ctrl+I` : comprendre du code

**Principe :** sélectionner du code existant → `Ctrl+I` → poser une question en langage naturel.

**Consigne :**
1. Sélectionner **tout le code** de la cellule ci-dessous
2. Appuyer sur `Ctrl+I`
3. Taper : `explique ce que fait ce code ligne par ligne`
4. Lire la réponse de Copilot dans la fenêtre inline

> **Objectif pédagogique :** utiliser Copilot comme outil de compréhension, pas seulement de génération.


In [ ]:
# Code à analyser avec Copilot (Ctrl+I → "explique ce que fait ce code")
import pandas as pd

df = pd.read_csv(ROOT / "data/raw/credit_risk_dataset.csv")

risk_by_grade = (
    df.groupby("loan_grade")["loan_status"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "taux_defaut", "count": "nb_prets"})
    .sort_values("taux_defaut", ascending=False)
    .round(3)
)
print(risk_by_grade)


---

### 🧪 Exercice 3 — Chat inline `Ctrl+I` : corriger une erreur

**Principe :** le code ci-dessous contient **3 erreurs volontaires**.  
Utiliser Copilot pour les identifier et les corriger.

**Consigne :**
1. Sélectionner tout le code
2. `Ctrl+I` → taper : `ce code contient des erreurs, corrige-les`
3. Comparer la correction proposée par Copilot avec le code original
4. Accepter si la correction est correcte — **vérifier avant d'accepter**

> **Erreurs cachées :** une faute de colonne, un mauvais type, une logique inversée.


In [8]:
revenu_moyen = df["person_income"].mean()
hauts_revenus = df[df["person_income"] > 50000]
taux_sains = 1 - df["loan_status"].mean()

print(f"Revenu moyen : {revenu_moyen:.0f}")
print(f"Hauts revenus : {len(hauts_revenus)} lignes")
print(f"Taux de prêts sains : {taux_sains:.1%}")


Revenu moyen : 66075
Hauts revenus : 18499 lignes
Taux de prêts sains : 78.2%


---

### 🧪 Exercice 4 — Chat inline `Ctrl+I` : refactorer du code

**Principe :** améliorer du code fonctionnel mais peu lisible.

**Consigne :**
1. Sélectionner tout le code
2. `Ctrl+I` → taper : `refactorise ce code pour le rendre plus lisible et ajoute des commentaires`
3. Observer comment Copilot restructure sans changer le comportement


In [ ]:
import pandas as pd
df = pd.read_csv(ROOT / "data/raw/credit_risk_dataset.csv")
# Code à refactorer — fonctionnel mais peu lisible
result = df[df["loan_status"]==1].groupby("loan_grade")["loan_amnt"].mean().reset_index().rename(columns={"loan_amnt":"montant_moyen_defaut"}).sort_values("montant_moyen_defaut",ascending=False).round(0)
print(result)


---

### 🧪 Exercice 5 — Chat latéral `Ctrl+Alt+I` : générer une fonction avec docstring

**Principe :** le chat latéral permet de décrire une fonction complexe en langage naturel.  
Copilot génère le code complet, la docstring, et parfois les tests.

**Consigne :**
1. Ouvrir le chat latéral : `Ctrl+Alt+I`
2. Coller ce prompt exact :

```
Écris une fonction Python appelée `calculer_taux_defaut` qui :
- prend en paramètre un DataFrame pandas et un nom de colonne de segmentation
- retourne un DataFrame avec le taux de défaut (loan_status) par segment
- trié par taux décroissant
- inclut le nombre de prêts par segment
- ajoute une docstring complète en français
```

3. Copier le code généré dans la cellule ci-dessous
4. Tester la fonction sur `loan_grade` puis sur `person_home_ownership`


In [ ]:
import pandas as pd


def calculer_taux_defaut(df, colonne_segmentation):
    """
    Calcule le taux de défaut par segment d'un DataFrame.

    Cette fonction regroupe les prêts selon une colonne fournie en entrée,
    calcule le taux moyen de la cible binaire `loan_status` pour chaque segment,
    puis retourne un DataFrame trié par taux de défaut décroissant.
    Elle inclut également le nombre total de prêts observés dans chaque segment.

    Paramètres
    ----------
    df : pandas.DataFrame
        DataFrame contenant au moins la colonne `loan_status` et la colonne
        de segmentation indiquée.
    colonne_segmentation : str
        Nom de la colonne utilisée pour créer les segments.

    Retourne
    -------
    pandas.DataFrame
        DataFrame avec une ligne par segment et les colonnes suivantes :
        - la colonne de segmentation
        - `taux_defaut` : proportion de prêts en défaut dans le segment
        - `nb_prets` : nombre total de prêts dans le segment

    Levée d'exceptions
    ------------------
    KeyError
        Si la colonne de segmentation ou la colonne `loan_status` est absente.
    """
    if colonne_segmentation not in df.columns:
        raise KeyError(f"La colonne '{colonne_segmentation}' est introuvable.")
    if "loan_status" not in df.columns:
        raise KeyError("La colonne 'loan_status' est introuvable.")

    resultat = (
        df.groupby(colonne_segmentation, dropna=False)["loan_status"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "taux_defaut", "count": "nb_prets"})
        .reset_index()
        .sort_values("taux_defaut", ascending=False)
    )

    resultat["taux_defaut"] = resultat["taux_defaut"].round(3)
    return resultat


# ── Test de la fonction ──────────────────────────────────────────────────────
df = pd.read_csv(ROOT / "data/raw/credit_risk_dataset.csv")

# Test 1 : segmentation par grade
print(calculer_taux_defaut(df, "loan_grade"))

# Test 2 : segmentation par type de logement
print("\n")
print(calculer_taux_defaut(df, "person_home_ownership"))

---

## Bloc 7 — Charger et explorer le dataset fil rouge

Le dataset `credit_risk_dataset.csv` est le **fil rouge de toute la formation**.  
Il sera utilisé dans chaque session, du nettoyage jusqu'au déploiement en production.

> **Avant d'exécuter :** s'assurer que `credit_risk_dataset.csv` est bien dans `data/raw/`.


In [ ]:
import pandas as pd
import numpy as np

# ── Chargement ──────────────────────────────────────────────────────────────
df = pd.read_csv(ROOT / "data/raw/credit_risk_dataset.csv")

print(f"Lignes   : {df.shape[0]:,}")
print(f"Colonnes : {df.shape[1]}")
print()
print("Colonnes et types :")
for col in df.columns:
    print(f"  {col:<35} {str(df[col].dtype)}")


In [ ]:
# ── Aperçu des 5 premières lignes ───────────────────────────────────────────
df.head(5)


In [ ]:
# ── Statistiques descriptives ───────────────────────────────────────────────
df.describe().round(2)


In [ ]:
# ── Valeurs manquantes ──────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    "Manquants": missing,
    "Pourcentage (%)": missing_pct
}).sort_values("Manquants", ascending=False)

print("Valeurs manquantes par colonne :")
print(missing_df[missing_df["Manquants"] > 0])
print()
if missing_df["Manquants"].sum() == 0:
    print("✅ Aucune valeur manquante")
else:
    total = missing_df["Manquants"].sum()
    print(f"⚠️  {total} valeurs manquantes — traitement en J1S4 (Feature Engineering)")


In [ ]:
# ── Taux de défaut — chiffre de référence de toute la formation ─────────────
print("=" * 50)
print("Distribution de la variable cible : loan_status")
print("=" * 50)
print()
print(df["loan_status"].value_counts())
print()
print(df["loan_status"].value_counts(normalize=True).round(3))
print()
taux_defaut = df["loan_status"].mean()
print(f"Taux de défaut : {taux_defaut:.1%}")
print()
print("→ À retenir pour toute la formation : 21,8 %")
print("  0 = prêt sain  ·  1 = défaut de paiement")


---

## Bloc 8 — Push sur le fork et Pull Request vers l'organisation

Le repo existe déjà depuis le Bloc 2 (fork de `bankrisk-formation`).  
On a maintenant : structure, `requirements.txt`, `.gitignore`, `README.md`.  
Il reste à **tout pousser sur `origin`** puis à ouvrir une **PR vers `upstream`**.

---

### Étape 1 : Vérifier ce qui va être commité

Avant tout commit, vérifier que les données ne sont **pas** incluses.


In [ ]:
# ── Vérification du statut avant commit ────────────────────────────────────
import os

os.chdir(ROOT)
print(f"Répertoire : {ROOT}")
print()

print("=== Fichiers qui seront commités ===")
git_run("git status")
print()

# Vérification que data/ n'apparaît pas dans les fichiers trackés
import subprocess
result = subprocess.run("git status --porcelain", shell=True, capture_output=True, text=True)
data_tracked = [l for l in result.stdout.splitlines() if 'data/' in l and '.gitkeep' not in l]
if data_tracked:
    print("❌ ATTENTION — des fichiers data/ vont être commités :")
    for f in data_tracked:
        print(f"   {f}")
    print("   → Vérifier le .gitignore")
else:
    print("✅ Aucun fichier data/ tracké (hors .gitkeep) — .gitignore OK")


In [ ]:
# ── Commit et push sur origin (le fork) ────────────────────────────────────
# ⚠️  Exécuter APRÈS avoir vérifié git status ci-dessus

# git_run('git add .')
# git_run('git commit -m "feat: J1S1 - structure + requirements + gitignore + README"')
# git_run('git push origin main')   # origin = TON fork, pas l'org

print("Cellule prête — décommenter les 3 lignes pour exécuter.")


---

### Étape 2 : Ouvrir la Pull Request vers l'organisation

Une fois le push effectué :

1. Aller sur `https://github.com/TON_USERNAME/credit-risk-formation`
2. GitHub affiche une bannière **"Compare & pull request"** → Cliquer
3. Vérifier les cibles :
   - **base repository** : `bankrisk-formation/credit-risk-formation` · **base** : `main`
   - **head repository** : `TON_USERNAME/credit-risk-formation` · **compare** : `main`
4. Titre de la PR : `feat: J1S1 - setup + structure + README`
5. **Create pull request** → le formateur merge

```
TON fork (origin)                    Organisation (upstream)
─────────────────                    ───────────────────────
TON_USERNAME/                        bankrisk-formation/
  credit-risk-formation   ──PR──▶     credit-risk-formation
  branch: main                        branch: main
```

> **Règle :** on ne push **jamais** directement sur `upstream`.  
> Tout passe par une PR depuis `origin`.


---

## Bloc 9 — Générer le `README.md` avec Copilot

Le `README.md` est la **carte de visite** du repo GitHub.  
Il doit expliquer le projet, la structure, comment l'utiliser — en une lecture de 2 minutes.

On va utiliser **deux approches complémentaires** pour le générer avec Copilot :
1. **Prompt structuré** dans le chat latéral → Copilot génère le contenu complet
2. **Cellule Python** → on écrit le fichier directement depuis le notebook

---

### Étape 1 : Prompt Copilot (chat latéral `Ctrl+Alt+I`)

Ouvrir le chat latéral et coller ce prompt :

```
Génère un README.md professionnel en français pour un projet Data Science avec cette structure :

Nom du projet : Credit Risk Analysis
Description : Analyse du risque de crédit sur le dataset Kaggle Credit Risk (32 581 prêts, 
cible binaire loan_status : 0=sain, 1=défaut). Taux de défaut : 21.8%.

Stack technique : Python 3.10, Pandas, NumPy, Plotly, Scikit-learn, MLflow, DagsHub,
Hugging Face, PySpark, Delta Lake, Streamlit, GitHub Actions.

Structure du projet :
- data/raw/         → Dataset original (non versionné)
- data/processed/   → Données transformées (non versionné)
- notebooks/        → Notebooks Jupyter par session
- src/              → Scripts Python réutilisables
- tests/            → Tests unitaires pytest

Sections à inclure :
1. Badge Python version, licence CC0
2. Description du projet et contexte bancaire
3. Structure du repo (arborescence ASCII)
4. Installation (clone + pip install -r requirements.txt)
5. Utilisation (lancer un notebook, lancer les tests)
6. Dataset (description + source Kaggle)
7. Stack technique (tableau par catégorie)
8. Roadmap (3 jours de formation)
9. Licence
```

> Observer comment Copilot structure le README complet.  
> On peut ensuite demander des ajustements avec `Ctrl+I` sur des sections spécifiques.


### Étape 2 : Écrire le README depuis Python

La cellule ci-dessous génère le `README.md` directement.  
Remplacer `TON_USERNAME` par ton nom GitHub avant d'exécuter.


In [ ]:
# ── Génération du README.md ──────────────────────────────────────────────────
# Remplacer TON_USERNAME par ton identifiant GitHub
GITHUB_USERNAME = "TON_USERNAME"
REPO_NAME = f"credit-risk-{GITHUB_USERNAME}"

readme_content = f"""# 🏦 Credit Risk Analysis

![Python](https://img.shields.io/badge/python-3.10-blue)
![Licence](https://img.shields.io/badge/licence-CC0-green)
![Status](https://img.shields.io/badge/status-en%20cours-yellow)

Analyse du risque de crédit sur le dataset Kaggle **Credit Risk Dataset**.
Projet fil rouge de la formation *Data Science with Python* — 3 jours.

> **Taux de défaut de référence : 21,8 %**  
> 32 581 prêts · 12 variables · Cible binaire `loan_status` (0=sain, 1=défaut)

---

## 📁 Structure du projet

```
{REPO_NAME}/
├── data/
│   ├── raw/            ← Dataset original — jamais modifié (non versionné)
│   └── processed/      ← Données nettoyées et features (non versionné)
├── notebooks/          ← Notebooks Jupyter par session de formation
├── src/                ← Scripts Python réutilisables
├── tests/              ← Tests unitaires pytest
├── requirements.txt    ← Dépendances Python
└── .gitignore          ← Données et secrets exclus
```

---

## ⚙️ Installation

```bash
# 1. Cloner le repo
git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
cd {REPO_NAME}

# 2. Créer et activer le virtualenv
python -m venv .venv
source .venv/bin/activate  # Mac/Linux
# .venv\\Scripts\\activate  # Windows

# 3. Installer les dépendances
pip install -r requirements.txt
```

---

## 📊 Dataset

| Propriété | Valeur |
|-----------|--------|
| **Source** | [Kaggle — Credit Risk Dataset](https://www.kaggle.com/datasets/laotse/credit-risk-dataset) |
| **Licence** | CC0 (domaine public) |
| **Lignes** | 32 581 |
| **Colonnes** | 12 |
| **Cible** | `loan_status` — 0 = sain, 1 = défaut |
| **Taux de défaut** | 21,8 % |

> Le fichier CSV doit être placé manuellement dans `data/raw/` — il n'est pas versionné.

---

## 🛠️ Stack technique

| Catégorie | Outils |
|-----------|--------|
| **Data** | Pandas, NumPy |
| **Visualisation** | Plotly |
| **Machine Learning** | Scikit-learn |
| **MLOps** | MLflow, DagsHub |
| **LLM** | Hugging Face (Mistral-7B-Instruct) |
| **Data Engineering** | PySpark, Delta Lake |
| **Application** | Streamlit |
| **CI/CD** | GitHub Actions |

---

## 🗓️ Roadmap — 3 jours de formation

### Jour 1 — Data Science with Python
- [x] J1S1 · Setup & GitHub Copilot
- [ ] J1S2 · Pandas & NumPy
- [ ] J1S3 · EDA & Plotly
- [ ] J1S4 · Feature Engineering

### Jour 2 — Advanced Machine Learning
- [ ] J2S1 · K-Means Risk Profiling
- [ ] J2S2 · Logistic Regression Credit Scoring
- [ ] J2S3 · MLflow & DagsHub
- [ ] J2S4 · LLM Explanations (Hugging Face)

### Jour 3 — Data Engineering
- [ ] J3S1 · Streamlit Dashboard
- [ ] J3S2 · PySpark SQL ETL
- [ ] J3S3 · Spark Structured Streaming
- [ ] J3S4 · Delta Lake & CI/CD

---

## 📄 Licence

Dataset sous licence **CC0 (Creative Commons Zero)** — domaine public, utilisation libre.  
Code source sous licence **MIT**.
"""

# Écriture à la RACINE du projet (pas dans notebooks/)
with open(ROOT / "README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("✅ README.md généré")
print(f"   Taille : {len(readme_content):,} caractères")
print()
# Aperçu des 20 premières lignes
lines = readme_content.split("\n")
for line in lines[:20]:
    print(line)
print("...")


### Étape 3 : Personnaliser avec Copilot `Ctrl+I`

Une fois le README généré, on peut l'affiner section par section.

**Exercices de personnalisation :**

| Sélectionner | Prompt Copilot (`Ctrl+I`) |
|---|---|
| La section Description | `rends cette description plus percutante pour un recruteur bancaire` |
| Le tableau Stack | `ajoute une colonne 'version' avec les numéros de version de requirements.txt` |
| La section Roadmap | `transforme cette roadmap en tableau avec les dates et le statut` |
| L'ensemble du README | `traduis en anglais en gardant le même format` |

> **Conseil :** tester au moins 2 de ces prompts pour voir comment Copilot modifie le contenu existant.


In [ ]:
# ── Vérification du README généré ───────────────────────────────────────────
from pathlib import Path

readme_path = ROOT / "README.md"

if readme_path.exists():
    content = readme_path.read_text(encoding="utf-8")
    lines = content.split("\n")
    sections = [l for l in lines if l.startswith("## ")]

    print(f"✅ README.md trouvé")
    print(f"   {len(lines)} lignes · {len(content):,} caractères")
    print()
    print("Sections détectées :")
    for s in sections:
        print(f"  {s}")
else:
    print("❌ README.md non trouvé — exécuter la cellule précédente")


### Étape 4 : Committer le README

```bash
git add README.md
git commit -m "docs: ajout README.md généré avec Copilot"
git push origin main
```

> **Résultat attendu :** sur github.com, la page d'accueil du repo affiche automatiquement le `README.md` rendu en HTML.  
> C'est la première chose que voit un recruteur ou un collaborateur qui visite le repo.


---

## ✅ Checklist de fin de session

| | Étape |
|---|-------|
| ☐ | Environnement opérationnel (VS Code **ou** Colab) — Python, Pandas, NumPy OK |
| ☐ | Repo `bankrisk-formation/credit-risk-formation` accessible sur github.com |
| ☐ | Fork `TON_USERNAME/credit-risk-formation` créé |
| ☐ | Fork cloné — `git remote -v` affiche `origin` **ET** `upstream` |
| ☐ | Structure créée à la racine : `data/raw/`, `data/processed/`, `notebooks/`, `src/`, `tests/` |
| ☐ | `.gitkeep` présents dans chaque dossier vide |
| ☐ | `requirements.txt` généré à la racine |
| ☐ | `.gitignore` généré — données exclues, `.gitkeep` conservés |
| ☐ | Copilot activé — premier `Tab` accepté (VS Code) |
| ☐ | Exercice 1 : commentaire-intention → Copilot a généré le code |
| ☐ | Exercice 2 : `Ctrl+I` → explication du `groupby` |
| ☐ | Exercice 3 : `Ctrl+I` → 3 erreurs corrigées |
| ☐ | Exercice 4 : `Ctrl+I` → code refactorisé |
| ☐ | Exercice 5 : chat latéral → fonction `calculer_taux_defaut` générée et testée |
| ☐ | `credit_risk_dataset.csv` dans `data/raw/` |
| ☐ | `df.shape` = `(32581, 12)` ✅ |
| ☐ | Taux de défaut **21,8 %** affiché ✅ |
| ☐ | `README.md` généré par Copilot et pushé |
| ☐ | PR mergée vers `bankrisk-formation` |

---

## ➡️ J1S2 — Pandas & NumPy
`describe()` · `groupby()` · `value_counts()` · types de données · export `data/processed/credit_features_j1.parquet`
